In [ ]:
%load_ext autoreload
%autoreload 2

# Solved Exercise 2
"*Some friends of yours are working on techniques for coordinating groups of mobile robots. Each robot has a radio transmitter that it uses to communicate with a base station, and your friends find that if the robots get too close to one another, then there are problems with interference among the transmitters. So a natural problem arises: how to plan the motion of the robots in such a way that each robot gets to its intended destination, but in the process the robots don’t come close enough together to cause interference problems.*

*We can model this problem abstractly as follows. Suppose that we have an undirected graph G = (V , E), representing the floor plan of a building, and there are two robots initially located at nodes a and b in the graph. The robot at node a wants to travel to node c along a path in G, and the robot at node b wants to travel to node d. This is accomplished by means of a schedule: at each time step, the schedule specifies that one of the robots moves across a single edge, from one node to a neighboring node; at the end of the schedule, the robot from node a should be sitting on c, and the robot from b should be sitting on d.*

*A schedule is interference-free if there is no point at which the two robots occupy nodes that are at a distance ≤ r from one another in the graph, for a given parameter r. We’ll assume that the two starting nodes a and b are at a distance greater than r, and so are the two ending nodes c and d. Give a polynomial-time algorithm that decides whether there exists an interference-free schedule by which each robot can get to its destination.*" (pg. 105)

### Discussion
We should begin by comming up with concrete scenarios.

We will asume the robots move in two dimensional space (aka a plane).

In the simplest scenario, the robots never have to walk in the direction towards the other's starting position in order to reach their destintion.

<figure>
<img src="./public/sol_ex_2/basic_space_2.png" width="300" alt="Fig. 1: 'a' and 'b' distance 'D' apart. 'c' *away* from 'b' in both axis. 'd' away from 'a' in both axis.">
<figcaption> Figure 1: When both targets are *strictly* away from the opposite robot, the schedules may never conflict.</figcaption>
</figure>

If *a* has to move in the positive 'x' direction and positive 'y' direction to get to *c*, and *b* has to move in the positive 'x' direction but negative 'y' direction to get to *d*, then they will only ever get more apart than their initial distance apart *d*.

We can speak of the straight-line segments that represent an idealized path from *a* to *c*, 'AC', and the same idealized path from *b* to *d* 'BD'.
In the previous example AC and BD never cross. In fact, if we think of the direction vectors that describe the direction of AC and BD, dir_ac and dir_bd, we can say that AC and BD diverge. 

In a different example, AC and BD might be well described as parallel, and again, there would be no concern of interference.

If AC and BD were not parallel nor diverging, then there is a possibility of interference.

It is possible for AC and BD to be convergent and still have no interference, for example if *a* and *b* start more than D apart: 

<figure>
<img src="./public/sol_ex_2/basic_space_3.png" width="300" alt="Fig. 2: 'a' and 'b' distance 'D-prime' apart. 'c' and 'd' distance 'D' apart. D-prime is bigger than D. AC and BD converge into each other, but never come closer than D">
<figcaption> Figure 2: When both robots are initially more than D away from each other, the schedules may never conflict.</figcaption>
</figure>

Across both examples, the most important characteristic of the idealized paths is that there is no point in either path that is closer to any point on the other path than D.

So any non-trivial scenario must have at least two points F_1 and F_2 in each idealized line segment such that they are closer to each other than D. 

<figure>
<img src="./public/sol_ex_2/basic_space_4.png" width="300" alt="Fig. 3: 'a' and 'b' distance 'D' apart. 'c' and 'd' distance 'D' apart. Unlike previous examples, 'c' and 'd' are now &quot;in a row&quot;, and there are multiple points in which AC and BD are closer to each other than D. An earlier pair of such points is labeled F_1 and F_2 in red.">
<figcaption> Figure 3: When the idealized segment paths have at least one point closert than D, there is a possibility of interference.</figcaption>
</figure>


However, in all of our scenarios so far, we've only been considering "complete", infinite planes as our spaces. There might be **terrain features** that prevent a robot from following the idealized segment paths. These could be described as 'holes' in our space. As for the infinity of the planes, it just makes sense to have the space the robots can traverse be bounded. 

<figure>
<img src="./public/sol_ex_2/terrain_1.png" width="300" alt="Fig. 4: brown rectangular area with blue circle super imposed such that there is a fifth of the height of the rectangular area between the top bound of the brown area and the top of the circle. The circle extends about one fifth of it's own diameter beyond the bottom of the rectangular area. The rectangular area extends to the right and left of the circle by about a fourth its width. The points 'a', 'b', 'c', and 'd' are towards the corners of the rectangular area. In black, the line segment AC connectds 'a' and 'c', while a curved segment BD connects 'b' and 'd' by going around the blue area. In red, a dashed line labeld 'BD' connecteds 'b' and 'd' over the blue circle.">
<figcaption> Figure 4: There might be cases where the idealized line segments go over impassable terrain.</figcaption>
</figure>

In the above figure we can see a scenario in which there is an obstacle robots cannot go over, which prevents them from following the idealized segment paths. 

Such cases required the introduction of "real paths" which might differ from idealized linear paths. 


One important disctintion between our idealized segment paths and real paths, is that the later require a robot 'turn'. With our single linear segments robots would start moving in one direction and only possibly move back and forth in that same direction. With real paths, a robot may be moving in one direction DIR_1 at some point in the schedule, and in another direction DIR_2 (not strictly opposite to DIR_1) in another point in the schedule.

We have thus reached the point in which schedules become important: the fact that there are points at which each robots path is closer than D to the other's path does not imply interference, since that requires both robots to be at such points **simultaenously**. If one robot moves along its path near the other's or even over the other's path but all while at least D away from the other, there is still no interference. 

<figure>
<img src="./public/sol_ex_2/schedules_1.png" width="400" alt="Fig. 5: ">
<figcaption> Figure 5: Real paths (in green) intersect; but if a schedule is followed, the robots do not come into distance D of each other. </figcaption>
</figure>

In the previous diagram, the ideal linear segments are not viable because they seemingly go over the other robot's "interference area". Two possible paths in green instead show a solution that works if *a* moves to it's destination along such a path **first** and then *b*.






One naive solution at this point would be simply try having one robot "go first", and then the other once the first has reached its destination. We could use BFS to find a path from r_1 to t_1, and if there's no path we can say that there's no solution. If there is a path but there is interference, then we'd have to try moving r_2 first and then coming back to r_1.

The question would be where to move r_2 to. If it is possible to move r_2 to it's destination we might be tempted to do so. If we find that even after moving r_2 to its destination, r_1 has no path to t_1, we might feel inclined to say that there's no solution. But how do we know that there is no way to move r_2 other than to it's destination such that r_1 is able to reach t_1?

Well so far, we don't. As a matter of fact, there could even be a sequence of intermediate moves in which r_1 and r_2 move not even necessarily in the direction of their destinations, but just enough to allow each robot to clear terrain in turns before actually heading to their respective destinations. 

<figure>
<img src="./public/sol_ex_2/terrain_2_the_ring.png" width="600" alt="Fig. 6: 4 sub-diagrams related by three arrows representing state changes. In the first subdiagram the two robots are on a ring-like terrain with an opening at the top-most point of its circumference. Overlayed over the subdiagram are each robots interference-regions. These overlap over the exit point. In the following three subdiagrams, one robot first moves up the ring until they are as close as possible to the other's interference area. Then the other moves away and down the ring as much as possible to clear its interference are from the only possible exit, finally alowing the first robot to escape the ring terrain.">
<figcaption> Fig. 6: Two robots can clear a "ring" obstacle by making alternating moves until one of them escapes it.</figcaption>
</figure>

Figure six shows a possible terrain configuration which forces the robots to move in an alternating fashion in order to clear a ring-shaped piece of terrain.  Another possible more succint scenario could simply involve two simple intersecting corridors, with starting and target positions such that one robot has to temporarily "move out of the way" for the other robot to get to its destination, before it gets to its own target itself. 

<figure>
<img src="./public/sol_ex_2/terrain_3_the_corridors.png" width="600" alt="Fig. 7: ">
<figcaption> Fig. 7: Two robots can clear an "intersection" by letting making way for one and then the other.</figcaption>
</figure>

Again, with our naive solution (let's call it **naive_one**), the terrains in figures six and seven would result in seemingly no viable schedule, where it is reasonable there'd be one if we alternate the movements of each robot instead.

Our first non-naive solver then has to be capable of alternately moving both robots in order to find a viable solution. The question then becomes how does our solver realize what bot has to move and where to. 

We could start by picking any robot and checking if they have a viable path to their destination.

```pseudo python
def pathAvailable(G: Graph, target: Node, bot_1: int, bot_2: int):
    ''' Returns sequence of nodes to reach target node, None otherwise '''
    path = getPathFromTo(G, G.getBotNode(bot_1), target)
    if path is None:
        return None:
    else:
        return path

```
A viable path would be one that connects the given bot's current standing node and the target node without causing interference with the other bot.

```pseudo python
def getPathFromTo(G: Graph, from: Node, to: Node):
    ''' Returns list of nodes to traverse from "from" to "to", None if no path is found '''
    t: dict = dict()
    t[from] = None
    q: queue = list(from)
    d: set = set(from)
    while len(q) > 0:
        current_node = q.front
        neighbors = G[current_node]
        for neighbor in neighbors:
            if neighbor not in d:
                d.insert(neighbor)
                q.append(neighbor)
                t[neighbor] = current_node
        q.popleft()
    if t.find(to) is None:
        return None
    else:
        return pathFromParentDict(target, t)
```
This previous function is underspecified, as it doesn't consider whether a node in the graph would be under the area of interference of the other bot before adding it to its "path".

With coordinates this becomes trivial, since we'd just need to apply the distance formula between two nodes to find out if its smaller than the interference distance. 

As trivial as it is, this means that coordinates will have to be built into our Graphs if not at least our Nodes.

```pseudo python
def getPathFromTo(G: Graph, from: Node, to: Node, D):
    ... look above for full implementation
    while len(q) > 0:
        current_node = q.front
        neighbors = G[current_node]
        for neighbor in neighbors:
            if neighbor not in d and notInInteference(other_bot, neighbor, D):
                d.insert(neighbor)
                q.append(neighbor)
                t[neighbor] = current_node
    ... 

def notInInterference(bot_source: Node, subject_node: Node,):
    dist: int = distance(bot_source.x, bot_source.y, subject_node.x, subject_node.y)

    if dist > D:
        return true
    else:
        return false

```
Now that we can decide whether there is a viable path to a bot's target, we can think about what to do when there isn't a viable path. 

If there wasn't a viable path, it would be useful to know whether it was because the terrain makes it impossible, or because the current robot positions result in paths with interference. So perhaps instead of only returning whether there is a path or not, we also want to return whether there was at least one path that would be viable if there was no interference. 

Of course then if there is no path even obviating interference, then there is no schedule to find. On the other hand if there is a path but it is currently being blocked by interference, then a possible solution is moving the other bot so as to get rid of the interference. 

So we can speak about an "intermediary", provisional path which we want to clear of interference so that this robot can walk it. To clear the interference from the provisional path, we want the other robot to walk away from the interfered nodes so as to get rid of the interference.

From the other robot's point of view, we can search for all nodes that are not within interference distance of the initial robot's provisional path, and walk to one of these nodes first, such that the initial robot can then walk to their destination. Since we don't want to waste time and compute resources, we could well look for the first node that satisfies this requirement.



At this point, we already have a non-naive solution worth implementing with some pseudocode to start converting. However, if we go back to fig. 7, we'll realize that our solution is still underspecified, as we haven't really dealt with the scenario where the robot to the left finds a viable path to its target, gets to it, but then has to move out of it in order to let the other robot through.

This scenario might feel "incorrect", since we have one robot making more-than-necessary movements in order to get both robots to their destination. In other words, such a schedule could be considered inefficient. Yet, this 'solved exercise' only asks us to find if a viable Schedule exists, not to find the shortest one by any means. Thus, it makes sense to go ahead and try to implement it as it is now. 

In [ ]:
from __future__ import annotations # pretty sure this is no longer needed starting 3.14
from dataclasses import dataclass
from typing import TypedDict
from myClasses.linkedList import LinkedList

@dataclass
class GraphNode:
    id: int
    location: tuple[int, int]

@dataclass
class Graph:
    '''This is the same Graph implementation we used for our "trees" earlier in this chapter.
    We will proceed to update it as necessary to solve this exercise.'''
    edges: list[tuple[int, int]]
    nodes: list[int]
    V: int = 0
    E: int = 0

    def __post_init__(self):
        V = len(self.nodes)
        E = len(self.edges)

@dataclass
class Bot:
    d: int
    pos: tuple[int, int]
    at_node: int

def findViableSchedule(G: Graph, bot_1: Bot, bot_2: Bot) -> bool:
    ''' Returns true if there is a schedule in which both (ro)Bots get to their
target destinations '''
    return True
    
INTERFERENCE_RANGE = 10
    
robot_one = Bot(INTERFERENCE_RANGE, (0, 0), 1)
robot_two = Bot(INTERFERENCE_RANGE, (0, 20), 2)

map_1 = Graph(edges = [(1,2),(1,3),(1,4),
                   (2,3),(2,4),
                   (3,4) # not as many edges in a non-directional graph...
                   ], nodes = [1,2,3,4])

findViableSchedule(map_1, robot_one, robot_two)

True